In [1]:
import sys
import os
import mysql.connector
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Ambil konfigurasi database dari folder utama project kalian
sys.path.append(os.path.abspath('..'))
from config import get_db_config

config = get_db_config()

# 1. Koneksi ke Database Baru (Fase Migrasi Sekarang)
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)

# 2. Koneksi ke Database Masa Depan (DB_FUTURE)
# Catatan: Pastikan di file config.py kalian sudah ada key 'db_future' ya!
db_future = mysql.connector.connect(**config['db_future'])
cursor_future = db_future.cursor(dictionary=True)

print(f"✅ Sukses Terhubung ke Database Baru : {config['db_new']['database']}")
print(f"🚀 Sukses Terhubung ke DB_FUTURE     : {config['db_future']['database']}")

✅ Sukses Terhubung ke Database Baru : 3
🚀 Sukses Terhubung ke DB_FUTURE     : 3


In [2]:
tables_to_check = [
    # --- Bagian Cimut ---
    "karyawan", 
    "keluarga_karyawan", 
    "bidang_kategori", 
    "bidang_link",
    
    # --- Bagian Afrida ---
    "periode", 
    "parameter_nilai", 
    "kabupaten", 
    "kecamatan",
    
    # --- Bagian Hanif ---
    "division_user", 
    "model_has_roles", 
    "model_has_permissions", 
    "kelurahan"
]

In [3]:
# === Cell 2: Inspeksi Detektor Pintar dengan Prioritas Target Revisi di Atas ===
import pandas as pd
import numpy as np

print("================================================================================")
print(" 🕵️‍♂️ FORENSIK ARSITEKTUR KONTROL: DETEKTOR AUTO-SMART COMPILER (REVISI PRIORITY) 🕵️‍♂️ ")
print("================================================================================")

# List penampung data di memori untuk keperluan sorting visualisasi
revisi_tables_queue = []
identical_tables_queue = []

# --- TAHAP A: PROSES PEN ARIKAN DATA & EVALUASI STRUKTUR DI BELAKANG LAYAR ---
for table in tables_to_check:
    try:
        # 1. Ambil data asli dari DB_NEW untuk kebutuhan .info() dan sampel isi data
        query = f"SELECT * FROM `{table}`"
        df_real_data = pd.read_sql(query, db_new)
        
        # 2. Tarik Struktur Fisik Kolom dari DB_NEW
        query_new_struct = f"""
        SELECT 
            c.COLUMN_NAME AS 'Nama Kolom',
            c.COLUMN_TYPE AS 'Tipe Data MySQL',
            CONCAT(
                CASE WHEN c.IS_NULLABLE = 'YES' THEN '✅ NULL (Boleh Kosong)' ELSE '🛑 NOT NULL (Wajib Isi)' END,
                CASE WHEN c.EXTRA = 'auto_increment' THEN '\\n🚀 AUTO_INCREMENT' ELSE '' END
            ) AS 'Aturan Nullability & Increment',
            CASE 
                WHEN c.COLUMN_KEY = 'PRI' THEN '🔑 PRIMARY KEY (PK)'
                WHEN c.COLUMN_KEY = 'MUL' AND k.REFERENCED_TABLE_NAME IS NOT NULL THEN '🔗 FOREIGN KEY (FK)'
                WHEN c.COLUMN_KEY = 'MUL' THEN 'INDEX'
                ELSE '-'
            END AS 'Status Kunci',
            CASE 
                WHEN k.REFERENCED_TABLE_NAME IS NOT NULL THEN CONCAT(k.REFERENCED_TABLE_NAME, ' (', k.REFERENCED_COLUMN_NAME, ')')
                ELSE '-'
            END AS 'Rujukan Induk (FK Origin)',
            CASE WHEN c.DATA_TYPE = 'enum' THEN REPLACE(REPLACE(REPLACE(c.COLUMN_TYPE, 'enum(', ''), ')', ''), "'", "") ELSE '-' END AS 'Daftar Pilihan ENUM'
        FROM INFORMATION_SCHEMA.COLUMNS c
        LEFT JOIN INFORMATION_SCHEMA.KEY_COLUMN_USAGE k 
            ON c.TABLE_SCHEMA = k.TABLE_SCHEMA AND c.TABLE_NAME = k.TABLE_NAME AND c.COLUMN_NAME = k.COLUMN_NAME
        WHERE c.TABLE_SCHEMA = '{config['db_new']['database']}' AND c.TABLE_NAME = '{table}'
        ORDER BY c.ORDINAL_POSITION;
        """
        df_struct_new = pd.read_sql(query_new_struct, db_new).drop_duplicates(subset=['Nama Kolom'], keep='first').reset_index(drop=True)
        
        # Lacak tabel anak yang nge-FK ke PK tabel ini di DB_NEW
        pk_referenced_list = []
        for idx, row_skri in df_struct_new.iterrows():
            if row_skri['Status Kunci'] == '🔑 PRIMARY KEY (PK)':
                lookup_fk_query = f"""
                SELECT CONCAT(TABLE_NAME, ' (', COLUMN_NAME, ')') as relasi
                FROM INFORMATION_SCHEMA.KEY_COLUMN_USAGE
                WHERE REFERENCED_TABLE_SCHEMA = '{config['db_new']['database']}'
                  AND REFERENCED_TABLE_NAME = '{table}'
                  AND REFERENCED_COLUMN_NAME = '{row_skri['Nama Kolom']}';
                """
                df_relasi_luar = pd.read_sql(lookup_fk_query, db_new)
                if not df_relasi_luar.empty:
                    pk_referenced_list.append("\n".join(df_relasi_luar['relasi'].tolist()))
                else:
                    pk_referenced_list.append("-")
            else:
                pk_referenced_list.append("-")
        df_struct_new['Tabel Yang nge-FK (DB_NEW)'] = pk_referenced_list

        # 3. Tarik Struktur Fisik Kolom dari DB_FUTURE
        query_future_struct = f"""
        SELECT 
            c.COLUMN_NAME AS 'Nama Kolom',
            c.COLUMN_TYPE AS 'Tipe Data MySQL',
            CONCAT(
                CASE WHEN c.IS_NULLABLE = 'YES' THEN '✅ NULL (Boleh Kosong)' ELSE '🛑 NOT NULL (Wajib Isi)' END,
                CASE WHEN c.EXTRA = 'auto_increment' THEN '\\n🚀 AUTO_INCREMENT' ELSE '' END
            ) AS 'Aturan Nullability & Increment',
            CASE 
                WHEN c.COLUMN_KEY = 'PRI' THEN '🔑 PRIMARY KEY (PK)'
                WHEN c.COLUMN_KEY = 'MUL' AND k.REFERENCED_TABLE_NAME IS NOT NULL THEN '🔗 FOREIGN KEY (FK)'
                WHEN c.COLUMN_KEY = 'MUL' THEN 'INDEX'
                ELSE '-'
            END AS 'Status Kunci',
            CASE 
                WHEN k.REFERENCED_TABLE_NAME IS NOT NULL THEN CONCAT(k.REFERENCED_TABLE_NAME, ' (', k.REFERENCED_COLUMN_NAME, ')')
                ELSE '-'
            END AS 'Rujukan Induk (FK Origin)',
            CASE WHEN c.DATA_TYPE = 'enum' THEN REPLACE(REPLACE(REPLACE(c.COLUMN_TYPE, 'enum(', ''), ')', ''), "'", "") ELSE '-' END AS 'Daftar Pilihan ENUM'
        FROM INFORMATION_SCHEMA.COLUMNS c
        LEFT JOIN INFORMATION_SCHEMA.KEY_COLUMN_USAGE k 
            ON c.TABLE_SCHEMA = k.TABLE_SCHEMA AND c.TABLE_NAME = k.TABLE_NAME AND c.COLUMN_NAME = k.COLUMN_NAME
        WHERE c.TABLE_SCHEMA = '{config['db_future']['database']}' AND c.TABLE_NAME = '{table}'
        ORDER BY c.ORDINAL_POSITION;
        """
        df_struct_future = pd.read_sql(query_future_struct, db_future).drop_duplicates(subset=['Nama Kolom'], keep='first').reset_index(drop=True)
        
        # Lacak tabel anak yang nge-FK ke PK tabel ini di DB_FUTURE
        pk_referenced_list_future = []
        for idx, row_skri in df_struct_future.iterrows():
            if row_skri['Status Kunci'] == '🔑 PRIMARY KEY (PK)':
                lookup_fk_query_future = f"""
                SELECT CONCAT(TABLE_NAME, ' (', COLUMN_NAME, ')') as relasi
                FROM INFORMATION_SCHEMA.KEY_COLUMN_USAGE
                WHERE REFERENCED_TABLE_SCHEMA = '{config['db_future']['database']}'
                  AND REFERENCED_TABLE_NAME = '{table}'
                  AND REFERENCED_COLUMN_NAME = '{row_skri['Nama Kolom']}';
                """
                df_relasi_luar_future = pd.read_sql(lookup_fk_query_future, db_future)
                if not df_relasi_luar_future.empty:
                    pk_referenced_list_future.append("\n".join(df_relasi_luar_future['relasi'].tolist()))
                else:
                    pk_referenced_list_future.append("-")
            else:
                pk_referenced_list_future.append("-")
        df_struct_future['Tabel Yang nge-FK (DB_FUTURE)'] = pk_referenced_list_future

        # 4. Deep Comparison Kesamaan Jeroan Kolom dasar
        cols_to_compare = ['Nama Kolom', 'Tipe Data MySQL', 'Aturan Nullability & Increment', 'Status Kunci', 'Rujukan Induk (FK Origin)', 'Daftar Pilihan ENUM']
        
        is_structure_identical = False
        if not df_struct_new.empty and not df_struct_future.empty:
            is_structure_identical = df_struct_new[cols_to_compare].equals(df_struct_future[cols_to_compare])

        # Wadah paket data tabel untuk di-render nanti
        table_package = {
            'name': table,
            'df_real_data': df_real_data,
            'df_struct_new': df_struct_new,
            'df_struct_future': df_struct_future,
            'is_identical': is_structure_identical
        }

        # 🔥 FILTER SAKTI CIMUT: Pisahkan antrean, utamakan yang bermasalah (revisi) ke atas!
        if is_structure_identical:
            identical_tables_queue.append(table_package)
        else:
            revisi_tables_queue.append(table_package)
            
    except Exception as e:
        print(f"❌ Gagal menganalisis awal tabel `{table}`: {e}")

# --- TAHAP B: MULAI PEN TAMPILAN VISUALISASI BERDASARKAN ANT REAN PRIORITAS ---

# 🚨 1. KELOMPOK UTAMA (PALING ATAS): DAFTAR TABEL YANG WAJIB DIREVISI 🚨
if revisi_tables_queue:
    print("\n" + "!"*80)
    print(f"🚨 [🔥 REVISI PRIORITY BOARD] TERDETEKSI {len(revisi_tables_queue)} TABEL BERBEDA - HARUS SEGERA DIPERBAIKI!")
    print("!"*80)
    
    for pkg in revisi_tables_queue:
        print(f"\n================================================================================")
        print(f"⚠️  [STATUS: TARGET REVISI] TABEL: {pkg['name'].upper()}")
        print(f"================================================================================")
        
        print(f"📋 1. Karakteristik Wadah Pandas Dataframe (.info()):")
        pkg['df_real_data'].info()
        print("\n" + "-"*60)
        
        print(f"🔍 2. [TABEL REAL SAAT INI] Arsitektur Fisik Kolom pada DB_NEW:")
        display(pkg['df_struct_new'].style.set_properties(**{'white-space': 'pre-wrap', 'text-align': 'left'}))
        print("\n" + "-"*60)
        
        print(f"🔮 3. [⚠️ TARGET BLUEPRINT RUJUKAN] Tinjau Struktur yang Benar pada DB_FUTURE di bawah:")
        if pkg['df_struct_future'].empty:
            print("❌ ERROR: Tabel ini tidak ditemukan / belum dibuat sama sekali di DB_FUTURE!")
        else:
            display(pkg['df_struct_future'].style.set_properties(**{'white-space': 'pre-wrap', 'text-align': 'left'}))
        print("\n" + "-"*60)
        
        print(f"📸 4. Sampel Isi Data Real di DB_NEW:")
        display(pkg['df_real_data'] if pkg['df_real_data'].empty else pkg['df_real_data'])
        print("\n" + "="*80)

# ✨ 2. KELOMPOK KEDUA (BAW AH): DAFTAR TABEL YANG SUDAH AMAN IDENTIK ✨
if identical_tables_queue:
    print("\n" + "="*80)
    print(f"✨ [🟢 PERFECTLY MATCHED BOARD] SEBANYAK {len(identical_tables_queue)} TABEL STRUKTURNYA SUDAH IDENTIK SAMA PERSIS!")
    print("="*80)
    
    for pkg in identical_tables_queue:
        print(f"\n================================================================================")
        print(f"✅ [STATUS: AMAN IDENTIK] TABEL: {pkg['name'].upper()}")
        print(f"================================================================================")
        
        print(f"📋 1. Karakteristik Wadah Pandas Dataframe (.info()):")
        pkg['df_real_data'].info()
        print("\n" + "-"*60)
        
        print(f"🔍 2. [TABEL REAL & FINAL] Arsitektur Fisik Kolom pada DB_NEW:")
        display(pkg['df_struct_new'].style.set_properties(**{'white-space': 'pre-wrap', 'text-align': 'left'}))
        print("\n" + "-"*60)
        
        print("✨ INDICATOR: [ DB_NEW ] & [ DB_FUTURE ] STRUKTUR SUDAH IDENTIK SAMA PERSIS 100% ✨")
        print("ℹ️  Tabel DB_FUTURE otomatis disembunyikan untuk kenyamanan pemandangan layar monitor.")
        print("\n" + "-"*60)
        
        print(f"📸 3. Sampel Isi Data Real di DB_NEW:")
        display(pkg['df_real_data'] if pkg['df_real_data'].empty else pkg['df_real_data'])
        print("\n" + "="*80)

 🕵️‍♂️ FORENSIK ARSITEKTUR KONTROL: DETEKTOR AUTO-SMART COMPILER (REVISI PRIORITY) 🕵️‍♂️ 

✨ [🟢 PERFECTLY MATCHED BOARD] SEBANYAK 12 TABEL STRUKTURNYA SUDAH IDENTIK SAMA PERSIS!

✅ [STATUS: AMAN IDENTIK] TABEL: KARYAWAN
📋 1. Karakteristik Wadah Pandas Dataframe (.info()):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51 entries, 0 to 50
Data columns (total 37 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id_karyawan           51 non-null     int64  
 1   id_user               51 non-null     object 
 2   kode_karyawan         51 non-null     object 
 3   nik_ktp               45 non-null     object 
 4   nama_lengkap          51 non-null     object 
 5   nama_panggilan        51 non-null     object 
 6   tempat_lahir          44 non-null     object 
 7   tanggal_lahir         45 non-null     object 
 8   jenis_kelamin         51 non-null     object 
 9   golongan_darah        31 non-null     object 
 10  agama  

,Nama Kolom,Tipe Data MySQL,Aturan Nullability & Increment,Status Kunci,Rujukan Induk (FK Origin),Daftar Pilihan ENUM,Tabel Yang nge-FK (DB_NEW)
0,id_karyawan,bigint(20) unsigned,🛑 NOT NULL (Wajib Isi) 🚀 AUTO_INCREMENT,🔑 PRIMARY KEY (PK),-,-,absensi (id_karyawan) catatan_kelas (id_karyawan) izin_karyawan (id_karyawan) karyawan_resign (id_karyawan) keluarga_karyawan (id_karyawan)
1,id_user,varchar(15),🛑 NOT NULL (Wajib Isi),-,-,-,-
2,kode_karyawan,varchar(20),✅ NULL (Boleh Kosong),-,-,-,-
3,nik_ktp,varchar(20),✅ NULL (Boleh Kosong),-,-,-,-
4,nama_lengkap,varchar(255),✅ NULL (Boleh Kosong),-,-,-,-
5,nama_panggilan,varchar(50),✅ NULL (Boleh Kosong),-,-,-,-
6,tempat_lahir,varchar(100),✅ NULL (Boleh Kosong),-,-,-,-
7,tanggal_lahir,date,✅ NULL (Boleh Kosong),-,-,-,-
8,jenis_kelamin,"enum('Laki laki','Perempuan')",✅ NULL (Boleh Kosong),-,-,"Laki laki,Perempuan",-
9,golongan_darah,varchar(5),✅ NULL (Boleh Kosong),-,-,-,-



------------------------------------------------------------
✨ INDICATOR: [ DB_NEW ] & [ DB_FUTURE ] STRUKTUR SUDAH IDENTIK SAMA PERSIS 100% ✨
ℹ️  Tabel DB_FUTURE otomatis disembunyikan untuk kenyamanan pemandangan layar monitor.

------------------------------------------------------------
📸 3. Sampel Isi Data Real di DB_NEW:


,id_karyawan,id_user,kode_karyawan,nik_ktp,nama_lengkap,nama_panggilan,tempat_lahir,tanggal_lahir,jenis_kelamin,golongan_darah,...,akun_instagram,akun_facebook,link_dokumen_pribadi,riwayat_kesehatan,tanggal_bergabung,keahlian,id_shift,status_aktif,foto_profile,ttd_digital
0,1,U00001,LEAP00102VI23,None,ADMINISTRATOR,ADMINISTRATOR,None,None,Perempuan,None,...,None,None,None,None,2026-01-01,None,2.0,1,logo.png,None
1,2,U00003,LEAP00313III23,3514186411980002,"Graciela Evanda Ronadi, S.Kom.",Graciela,Sidoarjo,1998-11-24,Perempuan,None,...,https://www.instagram.com/gracielaevr/,None,https://drive.google.com/drive/folders/1alw9Su...,"Maag, tipes",2026-01-01,None,2.0,1,1707279559_ec864cc58f50e8b890d5.jpg,1702002184_d82747bcbd0f2bdab0a7.png
2,3,U00011,LEAP01101XII20,3515135105910001,DANIAR AULIA RIZKI,DANIAR,SURABAYA,1991-05-11,Perempuan,B,...,@dar.od,tidak ada,https://drive.google.com/drive/u/0/folders/1va...,PREKLAMSIA,2026-01-01,None,2.0,0,1692700456_2d6352eb557d734e53b7.jpeg,None
3,4,U00012,LEAP01202III20,3578106705930001,Habibah Melyna,Habibah,Semarang,1993-05-27,Perempuan,AB,...,https://instagram.com/habibahmelyna?igshid=MzN...,None,None,"Alergi udang, pengawet makanan dan micin",2026-01-01,desain grafis,3.0,1,1686127161_1ec3d11da554fb668e7c.jpg,1702892302_89507e3f2d87fb0dfdee.png
4,5,U00014,LEAP01431VII18,3578035706820005,Laksmi Puspitowardhani,Laksmi,Surabaya,1982-06-17,Perempuan,O,...,https://www.instagram.com/laksmi_purplespace/?...,None,https://drive.google.com/drive/folders/1-7iI4-...,Liver,2026-01-01,None,3.0,1,1696338938_2a535db26e3f89919325.jpg,None
5,6,U00015,LEAP01514II11,None,Ika Asriani Yadin,Ika,None,None,Perempuan,None,...,None,None,None,None,2026-01-01,None,3.0,1,1694397431_30b88a94a51ff7c8024e.png,None
6,7,U00016,LEAP01619VI17,3524035806960001,"Luluk Fatikah Sari, S.Pd.",Luluk,Lamongan,1996-06-18,Perempuan,None,...,https://www.instagram.com/lulukfatikah/,None,https://drive.google.com/drive/folders/1CF3rrc...,Tipes dan sakit lambung,2026-01-01,None,3.0,1,1688370880_b894a9a6a8aa56770eaf.jpeg,1708307211_3e160b7a0d1560febc2c.png
7,8,U00018,LEAP01820IV21,3515165701920002,Ditari Kurnia,Tari,Surabaya,1992-01-17,Perempuan,B,...,ww,None,None,Maag dan darah rendah,2026-01-01,None,3.0,1,1742411840_4c3bd3fb30f609808130.shtml,None
8,9,U00019,LEAP01901VI19,3524094707820003,"Hartatik, S.S.",Tatik,Lamongan,1982-07-07,Perempuan,O,...,tati_bj (lupa password tidak bisa masuk ig mel...,Tati ((lupa password tidak bisa masuk facebook...,https://drive.google.com/drive/folders/1xy6IGd...,Tidak ada,2026-01-01,None,2.0,1,1690430190_e05a6e1833612e00b30a.jpeg,1708311112_ea63d8f1f2fb12dc6dbe.png
9,10,U00020,LEAP02030IV09,3578170506820004,Juni Arlianto,MJ,Surabaya,1982-06-05,Laki laki,B,...,7inchuuriki,None,https://drive.google.com/drive/folders/1-ZWjsg...,None,2026-01-01,None,2.0,1,1688629580_4df06718a9d1efc1946a.jpeg,None




✅ [STATUS: AMAN IDENTIK] TABEL: KELUARGA_KARYAWAN
📋 1. Karakteristik Wadah Pandas Dataframe (.info()):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64 entries, 0 to 63
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id_keluarga        64 non-null     int64 
 1   id_karyawan        64 non-null     int64 
 2   hubungan_keluarga  64 non-null     object
 3   nama_lengkap       64 non-null     object
 4   pekerjaan          64 non-null     object
 5   nomor_hp           64 non-null     object
dtypes: int64(2), object(4)
memory usage: 3.1+ KB

------------------------------------------------------------
🔍 2. [TABEL REAL & FINAL] Arsitektur Fisik Kolom pada DB_NEW:


...

,permission_id,model_type,model_id,id_division




✅ [STATUS: AMAN IDENTIK] TABEL: KELURAHAN
📋 1. Karakteristik Wadah Pandas Dataframe (.info()):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 83449 entries, 0 to 83448
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id_kelurahan    83449 non-null  int64 
 1   id_kecamatan    83449 non-null  int64 
 2   nama_kelurahan  83449 non-null  object
 3   kode_pos        0 non-null      object
dtypes: int64(2), object(2)
memory usage: 2.5+ MB

------------------------------------------------------------
🔍 2. [TABEL REAL & FINAL] Arsitektur Fisik Kolom pada DB_NEW:


,Nama Kolom,Tipe Data MySQL,Aturan Nullability & Increment,Status Kunci,Rujukan Induk (FK Origin),Daftar Pilihan ENUM,Tabel Yang nge-FK (DB_NEW)
0,id_kelurahan,bigint(20) unsigned,🛑 NOT NULL (Wajib Isi) 🚀 AUTO_INCREMENT,🔑 PRIMARY KEY (PK),-,-,calon_siswa (id_kelurahan) siswa (id_kelurahan)
1,id_kecamatan,bigint(20) unsigned,🛑 NOT NULL (Wajib Isi),🔗 FOREIGN KEY (FK),kecamatan (id_kecamatan),-,-
2,nama_kelurahan,varchar(100),🛑 NOT NULL (Wajib Isi),-,-,-,-
3,kode_pos,varchar(10),✅ NULL (Boleh Kosong),-,-,-,-



------------------------------------------------------------
✨ INDICATOR: [ DB_NEW ] & [ DB_FUTURE ] STRUKTUR SUDAH IDENTIK SAMA PERSIS 100% ✨
ℹ️  Tabel DB_FUTURE otomatis disembunyikan untuk kenyamanan pemandangan layar monitor.

------------------------------------------------------------
📸 3. Sampel Isi Data Real di DB_NEW:


,id_kelurahan,id_kecamatan,nama_kelurahan,kode_pos
0,1,1,Abian Tubuh Baru,None
1,2,1,Babakan,None
2,3,1,Bertais,None
3,4,1,Dasan Cermen,None
4,5,1,Mandalika,None
...,...,...,...,...
83444,83803,7308,Sawagumu,None
83445,83804,7309,Saoka,None
83446,83805,7309,Suprau,None
83447,83806,7309,Tampa Garam,None
